In [1]:
import sys
from pathlib import Path
import pandas as pd

try:
    PROJECT_ROOT = Path(__file__).resolve().parent.parent
except NameError:
    PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.core.trainer import Trainer
from src.model.svm_classifier import SVMClassifierModel

In [2]:
DATA_DIR = PROJECT_ROOT / "data" / "processed" / "linear"
train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

print("=== Data Loaded ===")
print(f"Train shape: {train_df.shape}, Test shape: {test_df.shape}")

=== Data Loaded ===
Train shape: (2370, 36), Test shape: (593, 36)


In [3]:
svm_model = SVMClassifierModel()

# Initialize the Trainer
# n_iter=20 is chosen as a balance between search coverage and computation time 
# for the custom SMO solver. cv=10 matches the project standard.
trainer = Trainer(
    model=svm_model, 
    n_iter=20, 
    cv=10, 
    random_state=42
)

# Run the tuning process (SMOTE -> RandomizedSearchCV)
trainer.tune(train_df)

# Evaluate on the unseen test set
trainer.evaluate_test(test_df)


[Trainer] Tuning SVMClassifierModel...
  CV=10, n_iter=20, scorer=F2-macro
  Class distribution: {np.int64(0): np.int64(316), np.int64(1): np.int64(1595), np.int64(2): np.int64(459)}
  SMOTE k_neighbors : 5
Fitting 10 folds for each of 20 candidates, totalling 200 fits

  Best params     : {'tol': 0.0001, 'max_passes': 5, 'kernel': 'poly', 'gamma': 0.1, 'degree': 3, 'coef0': 0.0, 'C': 0.1}
  Best CV F2-macro: 0.6837
  Train F2-macro  : 0.7516
  ✅ Overfit gap=0.0678 — ổn

[Test] F2-macro    : 0.7176
[Test] F1-weighted : 0.7501


{'f2_macro': 0.7175715494214646, 'f1_weighted': 0.7501107021765704}

In [4]:
best_params = trainer.model.best_params
test_f2 = trainer.test_metrics.get('f2_macro', 0)

print("\n=== Experiment Summary ===")
print(f"Best Kernel: {best_params.get('kernel')}")
print(f"Best Params: {best_params}")
print(f"Test F2-macro: {test_f2:.4f}")


=== Experiment Summary ===
Best Kernel: poly
Best Params: {'tol': 0.0001, 'max_passes': 5, 'kernel': 'poly', 'gamma': 0.1, 'degree': 3, 'coef0': 0.0, 'C': 0.1}
Test F2-macro: 0.7176
